In [0]:
-- CUSTOMER ANALYSIS.
-- 1. DISTRIBUTION OF CUSTOMERS IN VARIOUS CITY
USE olist_retail.gold_dataset;
CREATE OR REPLACE TABLE olist_retail.gold_dataset.CUSTOMER_COUNT_DISTRIBUTION AS
SELECT
    CUSTOMER_CITY,
    COUNT(DISTINCT CUSTOMER_UNIQUE_ID) AS CUSTOMER_COUNT,
    ROUND(
        COUNT(DISTINCT CUSTOMER_UNIQUE_ID) * 100.0 /
        SUM(COUNT(DISTINCT CUSTOMER_UNIQUE_ID) ) OVER(),
        2
    ) AS PERCENTAGE_SHARE
FROM dim_customer
GROUP BY customer_city 
ORDER BY customer_count DESC;

-- MONTHLY REVENUE GROWTH
CREATE OR REPLACE TABLE olist_retail.gold_dataset.MOM_GROWTH
WITH MONTHLY_REVENUE AS (
    SELECT
       MONTH(ORDER_DATE) AS CURRENT_MONTH,
       SUM(payment_value) AS REVENUE
    FROM FACT_SALES
    GROUP BY current_month
    ORDER BY current_month
)
SELECT
    CURRENT_MONTH,
    REVENUE,
    LAG(revenue, 1, 0) OVER (
        ORDER BY current_month
    ) AS previous_month_revenue,
    revenue - previous_month_revenue AS GROWTH
FROM monthly_revenue
ORDER BY current_month;

-- Churn Summary
CREATE OR REPLACE TABLE olist_retail.gold_dataset.churn_summary AS
WITH customer_last_order AS (
    SELECT
        customer_unique_id,
        MAX(order_date) AS last_order_date
    FROM fact_sales
    GROUP BY customer_unique_id
),
CUSTOMER_RECENCY AS (
SELECT
    customer_unique_id,
    last_order_date,
    DATEDIFF((SELECT MAX(order_date) FROM FACT_SALES), last_order_date) AS recency_days
FROM customer_last_order
)
SELECT
customer_unique_id,
    last_order_date,
    recency_days,
    CASE
        WHEN recency_days <= 30 THEN 'Active'
        WHEN recency_days <= 90 THEN 'At Risk'
        ELSE 'Churned'
    END AS churn_status
FROM customer_recency;

-- CLV ANALYSIS
CREATE OR REPLACE TABLE olist_retail.gold_dataset.CLV_SUMMARY
SELECT
    customer_unique_id,
    count(order_id) AS TOTAL_ORDERS,
    SUM(payment_value) AS TOTAL_REVENUE,
    AVG(payment_value) AS avg_order_value,
    CASE
    WHEN total_revenue < 1000 THEN 'Low Value'
    WHEN total_revenue < 5000 THEN 'Medium Value'
    ELSE 'High Value'
END AS CUSTOMER_TYPE
FROM fact_sales
GROUP BY customer_unique_id
ORDER BY total_revenue DESC;